# Apollo Phase 0 — Proof of Concept

Train a small model on MAESTRO and demonstrate offline music generation.

Steps:
1. Preprocess MAESTRO → token sequences
2. Train ApolloModel (small config)
3. Generate a response to an input melody
4. Export as MIDI

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path.home() / 'Projects' / 'apollo' / 'src'))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import json
import pretty_midi

from representation import (
    midi_to_events, events_to_tokens, tokens_to_events, events_to_midi,
    VOCAB_SIZE, TOKEN_OFFSETS
)
from model import ApolloModel

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {device}')

DATA_DIR = Path.home() / 'Projects' / 'apollo' / 'data' / 'raw' / 'maestro-v3.0.0'
OUTPUT_DIR = Path.home() / 'Projects' / 'apollo' / 'models'
MIDI_OUT_DIR = Path.home() / 'Projects' / 'apollo' / 'data' / 'processed'

In [ ]:
# Step 1: Preprocess MAESTRO → token sequences
# Process a subset for PoC (full training in Phase 1)

meta = pd.read_csv(DATA_DIR / 'maestro-v3.0.0.csv')
train_meta = meta[meta['split'] == 'train'].reset_index(drop=True)
print(f'Training files: {len(train_meta)}')

# For PoC, use first 100 files
POC_SIZE = 100
poc_meta = train_meta.iloc[:POC_SIZE]

all_token_seqs = []
errors = 0
for _, row in tqdm(poc_meta.iterrows(), total=len(poc_meta), desc='Tokenizing MIDI'):
    midi_path = DATA_DIR / row['midi_filename']
    try:
        events = midi_to_events(str(midi_path), max_events=1024)
        if len(events) < 20:
            continue
        tokens = events_to_tokens(events)
        all_token_seqs.append(tokens)
    except Exception as e:
        errors += 1
        if errors <= 3:
            print(f'Error: {e}')

print(f'\nProcessed {len(all_token_seqs)} files ({errors} errors)')
total_tokens = sum(len(s) for s in all_token_seqs)
print(f'Total tokens: {total_tokens:,}')
print(f'Avg tokens per file: {total_tokens / len(all_token_seqs):.0f}')

In [ ]:
# Step 2: Create training dataset

class MusicTokenDataset(Dataset):
    """Dataset of fixed-length token windows from MIDI files."""
    def __init__(self, token_sequences, seq_len=256, stride=128):
        self.windows = []
        for seq in token_sequences:
            for start in range(0, len(seq) - seq_len, stride):
                self.windows.append(seq[start:start + seq_len + 1])  # +1 for target
    
    def __len__(self):
        return len(self.windows)
    
    def __getitem__(self, idx):
        window = self.windows[idx]
        x = torch.tensor(window[:-1], dtype=torch.long)
        y = torch.tensor(window[1:], dtype=torch.long)
        return x, y

SEQ_LEN = 256
dataset = MusicTokenDataset(all_token_seqs, seq_len=SEQ_LEN, stride=128)
print(f'Training windows: {len(dataset)}')

# Split into train/val (90/10)
val_size = max(1, len(dataset) // 10)
train_size = len(dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Step 3: Train the model

model = ApolloModel(
    vocab_size=VOCAB_SIZE,
    d_model=256,
    nhead=4,
    num_layers=4,
    max_seq_len=SEQ_LEN,
    user_embed_dim=0,  # No user embedding for PoC
    dropout=0.1,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,} ({n_params/1e6:.2f}M)')

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-5)

NUM_EPOCHS = 20
best_val_loss = float('inf')
train_losses = []
val_losses = []

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    epoch_loss = 0
    n_batches = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_train_loss = epoch_loss / n_batches
    train_losses.append(avg_train_loss)
    
    # Validate
    model.eval()
    val_loss = 0
    n_val = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
            val_loss += loss.item()
            n_val += 1
    
    avg_val_loss = val_loss / max(n_val, 1)
    val_losses.append(avg_val_loss)
    scheduler.step()
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), OUTPUT_DIR / 'apollo_poc.pt')
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1}/{NUM_EPOCHS}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}, lr={scheduler.get_last_lr()[0]:.6f}')

print(f'\nBest val loss: {best_val_loss:.4f}')
print(f'Model saved to {OUTPUT_DIR / "apollo_poc.pt"}')

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Train Loss')
ax.plot(val_losses, label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Apollo PoC Training')
ax.legend()
plt.tight_layout()
plt.savefig(str(Path.home() / 'Projects' / 'apollo' / 'docs' / 'poc_training.png'), dpi=150)
plt.show()

print(f'Final train loss: {train_losses[-1]:.4f}')
print(f'Final val loss: {val_losses[-1]:.4f}')

In [ ]:
# Step 4: Generate music!
# Load best model
model.load_state_dict(torch.load(OUTPUT_DIR / 'apollo_poc.pt', weights_only=True))
model.eval()

# Take a real melody from the dataset as a prompt
# Use first 20 events (~100 tokens) from a validation file
val_meta = meta[meta['split'] == 'validation'].reset_index(drop=True)
prompt_path = DATA_DIR / val_meta.iloc[0]['midi_filename']
print(f'Prompt source: {val_meta.iloc[0]["canonical_title"]}')
print(f'Composer: {val_meta.iloc[0]["canonical_composer"]}')

prompt_events = midi_to_events(str(prompt_path), max_events=20)
prompt_tokens = events_to_tokens(prompt_events)
# Replace EOS with SEP to signal "now respond"
prompt_tokens[-1] = TOKEN_OFFSETS['sep']

print(f'Prompt: {len(prompt_events)} events, {len(prompt_tokens)} tokens')
print(f'Prompt pitches: {[e.pitch for e in prompt_events[:10]]}...')

In [ ]:
# Generate 3 variations with different temperatures
temperatures = [0.7, 0.9, 1.1]
generated_files = []

for temp in temperatures:
    prompt_tensor = torch.tensor([prompt_tokens], dtype=torch.long, device=device)
    
    output_tokens = model.generate(
        prompt_tensor,
        max_new_tokens=500,  # ~100 events worth
        temperature=temp,
        top_k=40,
    )
    
    # Extract only the generated portion (after SEP)
    all_tokens = output_tokens[0].cpu().tolist()
    sep_idx = all_tokens.index(TOKEN_OFFSETS['sep']) if TOKEN_OFFSETS['sep'] in all_tokens else len(prompt_tokens) - 1
    generated_tokens = all_tokens[sep_idx + 1:]
    
    # Decode to events
    gen_events = tokens_to_events([TOKEN_OFFSETS['bos']] + generated_tokens)
    print(f'\nTemp={temp}: generated {len(gen_events)} events, {len(generated_tokens)} tokens')
    
    if gen_events:
        # Export prompt + generated as MIDI
        out_path = MIDI_OUT_DIR / f'apollo_poc_temp{temp}.mid'
        
        # Combine: prompt events, then generated events
        combined = prompt_events + gen_events
        events_to_midi(combined, str(out_path))
        generated_files.append(out_path)
        print(f'  Saved to {out_path}')
        
        # Also save just the response
        response_path = MIDI_OUT_DIR / f'apollo_poc_response_temp{temp}.mid'
        events_to_midi(gen_events, str(response_path))
        
        # Print some stats
        pitches = [e.pitch for e in gen_events]
        velocities = [e.velocity for e in gen_events]
        print(f'  Pitch range: {min(pitches)}-{max(pitches)}')
        print(f'  Velocity range: {min(velocities):.2f}-{max(velocities):.2f}')
        print(f'  First 10 pitches: {pitches[:10]}')

In [ ]:
# Visualize: piano roll of prompt vs generated
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Use temperature=0.9 generation
prompt_tensor = torch.tensor([prompt_tokens], dtype=torch.long, device=device)
output_tokens = model.generate(prompt_tensor, max_new_tokens=500, temperature=0.9, top_k=40)
all_tokens = output_tokens[0].cpu().tolist()
sep_idx = all_tokens.index(TOKEN_OFFSETS['sep']) if TOKEN_OFFSETS['sep'] in all_tokens else len(prompt_tokens) - 1
gen_events = tokens_to_events([TOKEN_OFFSETS['bos']] + all_tokens[sep_idx + 1:])

fig, ax = plt.subplots(figsize=(16, 6))

# Plot prompt events
t = 0
for e in prompt_events:
    t += e.delta_time
    rect = plt.Rectangle((t, e.pitch), e.duration, 0.8, 
                          color='steelblue', alpha=0.6 + 0.4 * e.velocity)
    ax.add_patch(rect)
prompt_end = t + prompt_events[-1].duration if prompt_events else 0

# Plot generated events
t = prompt_end + 0.1  # small gap
for e in gen_events:
    t += e.delta_time
    rect = plt.Rectangle((t, e.pitch), e.duration, 0.8,
                          color='coral', alpha=0.6 + 0.4 * e.velocity)
    ax.add_patch(rect)

# Separator line
ax.axvline(x=prompt_end, color='gray', linestyle='--', linewidth=2, label='Prompt | Response')

ax.set_xlim(0, t + 1)
all_pitches = [e.pitch for e in prompt_events + gen_events]
if all_pitches:
    ax.set_ylim(min(all_pitches) - 2, max(all_pitches) + 2)
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('MIDI Pitch')
ax.set_title('Apollo PoC — Prompt (blue) → Generated Response (coral)')

blue_patch = mpatches.Patch(color='steelblue', alpha=0.7, label='Input (prompt)')
coral_patch = mpatches.Patch(color='coral', alpha=0.7, label='Apollo response')
ax.legend(handles=[blue_patch, coral_patch])

plt.tight_layout()
plt.savefig(str(Path.home() / 'Projects' / 'apollo' / 'docs' / 'poc_piano_roll.png'), dpi=150)
plt.show()

print('\n=== Phase 0 PoC Complete ===')
print(f'Model: {sum(p.numel() for p in model.parameters()):,} params')
print(f'Trained on: {POC_SIZE} MAESTRO files')
print(f'Generated MIDI files in: {MIDI_OUT_DIR}')